In [36]:
import pandas as pd
import jellyfish

In [37]:
train_df = pd.read_parquet("./train.parquet")
val_df = pd.read_parquet("./val.parquet")
test_df = pd.read_parquet("./test.parquet")


In [38]:
print ("Our desired 1:0 ration should be 1:9 to better represent the true population")
for name, df in zip (["train","val","test"], [train_df, val_df, test_df]):
    print(f"{'='*10}{name}{'='*30}")
    distr = df["label"].value_counts()
    display(distr.head())
    total = distr[0] + distr[1]
    print(f"1:0 ratio {distr[1]/total:.3f}:,{distr[0]/total:.3f}")


Our desired 1:0 ration should be 1:9 to better represent the true population
==========train==============================


label
0    13592
1     1510
Name: count, dtype: int64

1:0 ratio 0.100:,0.900
==========val==============================


label
0    789
1     87
Name: count, dtype: int64

1:0 ratio 0.099:,0.901
==========test==============================


label
0    677
1     75
Name: count, dtype: int64

1:0 ratio 0.100:,0.900


In [39]:
import textdistance
import os

def _evaluation_function(s):
    #character-based
    jw_street_sim = jellyfish.jaro_winkler_similarity(s["StreetName_a"], s["StreetName_b"])
    jw_zipcode_sim =  jellyfish.jaro_winkler_similarity(s["ZipCode_a"], s["ZipCode_b"])
    jw_address_sim =  jellyfish.jaro_winkler_similarity(str(s["Address_a"]), str(s["Address_b"])) 
    lv_street_sim = 1 - jellyfish.levenshtein_distance(s["StreetName_a"], s["StreetName_b"])/(max(len(s["StreetName_a"]), len(s["StreetName_b"])))
    lv_zipcode_sim = 1 - jellyfish.levenshtein_distance(s["ZipCode_a"], s["ZipCode_b"])/max((len(s["ZipCode_a"]), len(s["ZipCode_b"])))
    lv_address_sim = 1 - jellyfish.levenshtein_distance(str(s["Address_a"]), str(s["Address_b"]))/max(len(str(s["Address_a"])), len(str(s["Address_b"])))
    hm_zipcode_sim = 1 - jellyfish.hamming_distance(s["ZipCode_a"], s["ZipCode_b"])/5

    #n-grams
    metrics = {
        "jaccard": (textdistance.Jaccard(qval = 2), textdistance.Jaccard(qval = 1)),
        "sorenson":(textdistance.Sorensen(qval = 2), textdistance.Sorensen(qval = 1)),
        "overlap": (textdistance.Overlap(qval=2), textdistance.Overlap(qval=1)),
        "tversky": (textdistance.Tversky(qval=2), textdistance.Tversky(qval=1)),
        "cosine": (textdistance.Cosine(qval = 2),textdistance.Cosine(qval = 1) )
    }

    n_gram_results = {}
    for option, (initial, for_zip) in metrics.items():
        street_score = initial.normalized_similarity(s["StreetName_a"], s["StreetName_b"])
        zip_score = initial.normalized_similarity(s["ZipCode_a"], s["ZipCode_b"])
        address_score = for_zip.normalized_similarity(str(s["Address_a"]), str(s["Address_b"]))
        n_gram_results[f"{option}_street"] = street_score
        n_gram_results[f"{option}_zip"] = zip_score
        n_gram_results[f"{option}_address"] = address_score

    return pd.Series(
        {
            **n_gram_results,
            "jw_street_sim": jw_street_sim,
            "jw_zipcode_sim": jw_zipcode_sim,
            "jw_address_sim": jw_address_sim,
            "lv_street_sim": lv_street_sim,
            "lv_zipcode_sim": lv_zipcode_sim,
            "lv_address_sim": lv_address_sim,
            "hm_zipcode_sim": hm_zipcode_sim,
        })
def _process_df(df):
    return pd.DataFrame([_evaluation_function(row) for _,row in df.iterrows()])


parquet_eval = [
    [train_df, "./evaluation_train"],
    [val_df, "./evaluation_val"],
    [test_df, "./evaluation_test"]
]

ret = []
for df,filepath in parquet_eval:
    if os.path.exists(filepath):
        curr = pd.read_parquet(filepath)
    else:
        curr = _process_df(df)
        curr.to_parquet(filepath, index = False)
    ret.append(curr)
if len(ret) != 3:
    raise IndexError("Did you modify parquet eval?")
X_train, X_val, X_test = ret

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

ret = []

model = LogisticRegression()
model.fit(X_train, train_df["label"])
preds = model.predict(X_val)
probs = model.predict_proba(X_val)[:,1]
ret.append([preds, probs])

In [41]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)         # fit on train only
svc = SVC()                                    # default kernel="rbf"
svc.fit(scaler.transform(X_train), train_df["label"])
preds = svc.predict(scaler.transform(X_val))
probs = svc.decision_function(scaler.transform(X_val))
ret.append([preds, probs])

In [42]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=0)    # n_estimators=100 by default
rf.fit(X_train, train_df["label"])
preds = rf.predict(X_val)
probs = rf.predict_proba(X_val)[:, 1]
rf.feature_importances_
ret.append([preds, probs])

In [43]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=0)            # sklearn-style wrapper; you know these hyperparameters from the side lab
xgb.fit(X_train, train_df["label"])
preds = xgb.predict(X_val)
probs = xgb.predict_proba(X_val)[:, 1]
ret.append([preds, probs])

In [57]:
from sklearn.metrics import classification_report

# probs
# model.classes_
# print(pd.Series(model.coef_[0], index = X_train.columns))
#Logistic Regression, SVM, Random Forest, XGBoost

models = ["Logistic Regression", "SVM", "Random Forest", "XGBoost"]
for i, (preds, probs) in enumerate(ret):
    print(f"{'='*20} {models[i]} {'='*20}")
    print(classification_report(val_df["label"], preds))
    print([i for i in range(len(preds)) if val_df["label"][i] != preds[i]])

    wrong_comparison = val_df[preds != val_df["label"].values][["StreetName_a", "Address_a","ZipCode_a","street_extension_a", "StreetName_b", "Address_b", "ZipCode_b", "street_extension_b", "label"]].rename(columns = {"label": "correct_label"}).sort_values(by = "correct_label")

    wrong_guess = wrong_comparison[["correct_label"]].apply(
        lambda a: a^1
    ).rename(columns = {"correct_label": "incorrect_label"})

    comparison_df = pd.concat([wrong_comparison, wrong_guess], axis = 1)
    print(
        comparison_df
    )

==================== Logistic Regression ====================
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       789
           1       0.93      0.94      0.94        87

    accuracy                           0.99       876
   macro avg       0.96      0.97      0.97       876
weighted avg       0.99      0.99      0.99       876

[2, 119, 140, 263, 351, 422, 530, 598, 630, 663, 824]
    StreetName_a  Address_a ZipCode_a street_extension_a StreetName_b  \
140         DEAN         87     06902                 ST          DEN   
351  WILLOWBROOK         64     06902                AVE    ILL BROOK   
422     GLENDALE         11     06906                CIR     GLEEDALE   
530      HACKETT         16     06906                CIR      HAMKETT   
598    SOUNDVIEW         95     06902                 DR    SOUNDVIEW   
630     WESTOVER         99     06902                 RD       WESTER   
2       NEWFIELD        658     06905         